# Transition to Per‑Patch Device Allocations (with Halos)

Our next step is moving from a single global managed memory allocation to a more fine-grained approach.
* Host memory remains **one single array** to avoid breaking compatibility with initialization and file I/O functions.
* Each patch will have a **dedicated allocation on its owning GPU**.
* Each patch's device allocation includes **halo layers**.

The below figures illustrate the approach on a conceptual level.

<table>
  <tr>
    <td align="center">
      <img src="./img/halos-step-0.png" height="320" alt="Halo Visualization Step 0"><br>
      <div>Partitioning of the global domain into two (or any other number of) patches.</div>
    </td>
    <td align="center">
      <img src="./img/halos-step-1.png" height="320" alt="Halo Visualization Step 1"><br>
      <div>Separate arrays for each patch, each extended by halo layers.</div>
    </td>
    <td align="center">
      <img src="./img/halos-step-2.png" height="320" alt="Halo Visualization Step 2"><br>
      <div>Patches now overlap - each halo is logically equivalent to an inner row on another patch.</div>
    </td>
    <td align="center">
      <img src="./img/halos-step-3.png" height="320" alt="Halo Visualization Step 3"><br>
      <div>The approach can be applied to an arbitrary number of patches (up to the number of inner rows).</div>
    </td>
  </tr>
</table>

Implementing this approach is slightly more involved than what we have done before.
Let's break down the necessary steps.

### 1. Extend the Patch Struct

Each patch now needs to know its local number of cells per dimension `localNumCellsX` and `localNumCellsY`.
For convenience, we also store the total size of each of the allocated arrays (`localSize`) which can be used in allocations and copy operations.
Since device data is now unique to each patch, we add `d_localU` and `d_localUNew` which hold the device pointers to the patch's arrays.

```cpp
struct Patch {
    // ... as before

    // patch local extents including halos/ boundaries
    size_t localNumCellsX;
    size_t localNumCellsY;
    size_t localSize;          // in bytes

    // pointers to the GPU allocation
    double* d_localU;
    double* d_localUNew;

    // ... as before
};
```


### 2. Compute Local Extents

We extend our initialization code to compute the new patch parameters.

```cpp
patch.localNumCellsX = patch.globalInnerEndX - patch.globalInnerBeginX + 2;   // two halo layers of size one each
patch.localNumCellsY = patch.globalInnerEndY - patch.globalInnerBeginY + 2;
patch.localSize = patch.localNumCellsX * patch.localNumCellsY * sizeof(double);
```


### 3. Device Allocations and De-Allocation

While host allocations remain as before, device allocations now needs to be done per patch.
Note that we need to call `cudaSetDevice` to make sure to select the appropriate device for allocation.

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    checkCudaError(cudaSetDevice(patchIdx));

    auto &patch = patches[patchIdx];

    checkCudaError(cudaMalloc((void **)&patch.d_localU,    patch.localSize));
    checkCudaError(cudaMalloc((void **)&patch.d_localUNew, patch.localSize));
}
```


De-allocation works in the same way.

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    checkCudaError(cudaSetDevice(patchIdx));

    auto &patch = patches[patchIdx];

    checkCudaError(cudaFree(patch.d_localU));
    checkCudaError(cudaFree(patch.d_localUNew));
}
```


### 4. Fix Initial H2D Copy

To fill the per-patch device buffers, multiple copy operations are necessary.
Their source is a window in the global source array, while their destination is each patch's full device array.
Note that this will copy slightly more data than strictly necessary since the halos are copied unnecessarily.

```cpp
auto sourceIdx = (patch.globalInnerBeginY - 1) * globalNumCellsX;   // include halo row
checkCudaError(cudaMemcpy(
    patch.d_localU,                 // destination - local to the patch
    &u[sourceIdx],                  // source - one large host array
    patch.localSize,                // size including halos
    cudaMemcpyHostToDevice));       // direction
```


One easy optimization that we can apply at this stage is initializing the second array from the first one.
We do this with a straight-forwards device-to-device copy.

```cpp
// initialize uNew by copying from u
checkCudaError(cudaMemcpy(patch.d_localUNew, patch.d_localU, patch.localSize, cudaMemcpyDeviceToDevice));
```

This also allows us skipping having a `uNew` array on the CPU altogether.

In the case of our 1D decomposition, each patch region (including top/ bottom halo rows) is laid out contiguously in the host array, which facilitates the copy operation.
In other cases, it can be beneficial to use specialized functions, e.g. `cudaMemcpy2D` for strided copies of interior sub‑rectangles, or splitting the host array as well.

### 5. Adapt D2H Copies

For our device to host copies we employ a similar logic.
One difference, however, is that we exclude the top and bottom halo (or boundary) layers as a small optimization.
This also increases robustness in case halos hold stale values.

```cpp
checkCudaError(cudaMemcpy(
    &u[patch.globalInnerBeginY * globalNumCellsX],                      // destination - a slice in the global host array; skip halo row
    &patch.d_localU[1 * patch.localNumCellsX],                          // source - local to the patch; skip top halo
    (patch.localNumCellsY - 2) * patch.localNumCellsX * sizeof(double), // size excluding top and bottom halo rows (but including left/right halos)
    cudaMemcpyDeviceToHost));                                           // direction
```


This still (unnecessarily) transfers the left and right halos.
One alternative, which we won't pursue in this course, is using `cudaMemcpy2D`.
In case you are interested, an implementation could look like this:

```cpp
checkCudaError(cudaMemcpy2D(
    &u[patch.globalInnerBeginY * globalNumCellsX + patch.globalInnerBeginX],    // destination - slice of host array
    globalNumCellsX * sizeof(double),                                           // destination pitch including halos
    &patch.d_localU[1 * patch.localNumCellsX + 1],                              // source - local to the patch excluding halos
    patch.localNumCellsX * sizeof(double),                                      // source pitch including halos
    (patch.localNumCellsX - 2) * sizeof(double),                                // width of the 2D copy excluding halos in bytes
    patch.localNumCellsY - 2,                                                   // height of the 2D copy excluding halos in rows
    cudaMemcpyDeviceToHost));                                                   // direction
```

### 6. Adapt Kernel Launches

Our kernel launches now get considerably easier since each kernel only 'sees' its local allocation.
As result, we now execute the kernel with the iteration space from `[1, 1]` to `[patch.localNx - 1, patch.localNy - 1]`, and use the patch-local device pointers.

```cpp
stencil2D<<<patch.gridSize, patch.blockSize>>>(
    patch.d_localU, patch.d_localUNew,                  // local pointers
    1, patch.localNumCellsX - 1,                        // local x interval
    1, patch.localNumCellsY - 1,                        // local y interval
    patch.localNumCellsX                                // stride for index linearization
);
```


### 7. Fix Pointer Swaps

Since we have multiple arrays for the device data now, we need to swap each pair separately.

```cpp
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    auto &patch = patches[patchIdx];
    std::swap(patch.d_localU, patch.d_localUNew);
}
```

Note that we still need a global barrier across devices to make sure that 

### 8. Implement Halo Exchange

After each iteration's compute phase, we push the boundary inner rows from each patch to the respective neighbor's halo row.
The easiest way of doing this is pushing `uNew` by means of `cudaMemcpyAsync` in the **default stream**.
This works, because
* the values have already been produces for the source patch (kernel in the same stream), and
* the values in the halo layer of that array will not be read on the destination patch.

We start by spanning a lambda function to structure our code better.

```cpp
auto haloExchange = [&](int deviceIdx) {
    // exchange halos between patches
    const auto &patch = patches[deviceIdx];
```


Next, we push data to the lower neighbor - provided there is one.
The tricky part is getting the indices right:
* We write to the top halo layer of `lowerPatch` at the very last row (`lowerPatch.localNy - 1`).
* We read from `patch` at the first **inner** row (`1`).
* We copy one row (`patch.localNx * sizeof(double)` bytes). This could be optimized to skip the first and last (left and right) halo element.

```cpp
    // push data to lower neighbor
    if (deviceIdx > 0) {
        const auto &lowerPatch = patches[deviceIdx - 1];

        // top halo of lower neighbor
        auto destPtr = &lowerPatch.d_localUNew[(lowerPatch.localNumCellsY - 1) * lowerPatch.localNumCellsX];
        // first inner row of current patch
        auto srcPtr  = &patch.d_localUNew[1 * patch.localNumCellsX];

        auto size = patch.localNumCellsX * sizeof(double);  // size of one row

        checkCudaError(cudaMemcpyAsync(destPtr, srcPtr, size, cudaMemcpyDeviceToDevice));
    }
```


Then we do the same for the upper neighbor:
* We write to the top halo layer of `upperPatch` at the very first row (`0`).
* We read from `patch` at the last **inner** row (`patch.localNy - 2`).
* We still copy one row (`patch.localNx * sizeof(double)` bytes).

```cpp
    // push data to upper neighbor
    if (deviceIdx < numDevices - 1) {
        const auto &upperPatch = patches[deviceIdx + 1];

        // bottom halo of upper neighbor
        auto destPtr = &upperPatch.d_localUNew[0 * upperPatch.localNumCellsX];
        // last inner row of current patch
        auto srcPtr  = &patch.d_localUNew[(patch.localNumCellsY - 2) * patch.localNumCellsX];

        auto size = patch.localNumCellsX * sizeof(double);  // size of one row

        checkCudaError(cudaMemcpyAsync(destPtr, srcPtr, size, cudaMemcpyDeviceToDevice));
    }
};
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/07-halos/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/07-halos/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/07-halos/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/07-halos/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/07-halos/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/07-halos/stencil-2d-medium.cu ../src/07-halos/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/07-halos/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/07-halos/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/07-halos/stencil-2d-easier.cu ../src/07-halos/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/07-halos/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/07-halos/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/07-halos/stencil-2d-solution.cu ../src/07-halos/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [07-halos/stencil-2d.cu](../src/07-halos/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/07-halos ../src/07-halos/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/07-halos 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/07-halos $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/07-halos.out --error=../output/07-halos.err \
    --wrap="../build/07-halos $((32 * 1024)) 256 2 8 0"

cat ../output/07-halos.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs in the `--gres=gpu:a100:NGPU` to anything between one and eight GPUs.

The profile is then available at [profiles/07-halos.nsys-rep](../profiles/07-halos.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/07-halos-nsys.out --error=../output/07-halos-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/07-halos \
        ../build/07-halos $((32 * 1024)) 256 2 8 0"

cat ../output/07-halos-nsys.out

## Next Step

The next step is figuring out why there are synchronization effects in [07-02](./07-halos-2.ipynb).